**Step 2: Generate the Query Sets**

This notebook writes the searches the experiments run: for every validation and test person, the queries a visitor might type to find them, grouped in the five query sets of the experiment design. It reads `./outputs/step_1_persons.csv` and changes nothing else. Every random step is seeded, so a re-run writes the same queries. It outputs:

1. `./outputs/step_2_queries.csv`: one row per query, with its split, query set, text, target person, relevant persons, masked fields and tokens

**1. Configuration and Helpers**

This cell defines the settings, reads the persons, and copies the normalisation of step 1, which a query token needs once a typing error or a respelling has changed it.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and helpers  —  settings, persons and the normalisation
#
# A query's tokens keep the field (given or surname) and the normalised
# recorded token they come from (`source_norm`), which the partial-name
# set QS5 uses to decide who else is a correct answer.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
OUTPUTS_DIR    = Path("outputs")   # every step's outputs
SEED           = 333               # every random step
QS5_SIZE       = 300               # partial-name queries sampled from QS1–QS4
QS5_SURNAME_P  = 0.6               # chance that a QS5 query keeps the surname
RULE           = "─" * 72

FINAL_TO_MEDIAL   = str.maketrans("ךםןףץ", "כמנפצ")     # final letters → medial forms
MAQAF             = "\N{HEBREW PUNCTUATION MAQAF}"     # the Hebrew hyphen
HEBREW_LETTERS    = "\N{HEBREW LETTER ALEF}-\N{HEBREW LETTER TAV}"            # a regex range
HEBREW_MARKS      = re.compile("[\N{HEBREW ACCENT ETNAHTA}-\N{HEBREW POINT QAMATS QATAN}]")
HEBREW_NON_LETTER = re.compile(f"[^{HEBREW_LETTERS} ]")  # anything but a letter or space
YIDDISH_LIGATURES = {"\N{HEBREW LIGATURE YIDDISH DOUBLE VAV}": "וו",
                     "\N{HEBREW LIGATURE YIDDISH VAV YOD}": "וי",
                     "\N{HEBREW LIGATURE YIDDISH DOUBLE YOD}": "יי"}


def token_script(text: str) -> str:
    """Return `hebrew` if `text` holds any Hebrew character, else `latin`."""
    return "hebrew" if any(0x0590 <= ord(char) <= 0x05FF for char in text) else "latin"


def normalise(text: str) -> str:
    """Normalise one token in its own script, as step 1 does, without spaces."""
    if token_script(text) == "latin":
        decomposed = unicodedata.normalize("NFKD", text)
        return "".join(c for c in decomposed
                       if not unicodedata.combining(c) and c.isalpha()).lower()
    text = HEBREW_MARKS.sub("", unicodedata.normalize("NFKD", text).replace(MAQAF, " "))
    for ligature, letters in YIDDISH_LIGATURES.items():
        text = text.replace(ligature, letters)
    return HEBREW_NON_LETTER.sub("", text.translate(FINAL_TO_MEDIAL)).replace(" ", "")


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` (dictionaries with the same keys) to the CSV file `path`."""
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


with (OUTPUTS_DIR / "step_1_persons.csv").open(encoding="utf-8", newline="") as _file:
    persons = [p | {"tokens": json.loads(p["tokens"])} for p in csv.DictReader(_file)]
persons_by_id = {p["person_id"]: p for p in persons}
print(f"✓ Configuration and helpers loaded ({len(persons)} persons, seed {SEED}).")

✓ Configuration and helpers loaded (637 persons, seed 333).


**2. Query Sets QS1–QS4**

This cell takes `persons` and writes one query per validation and test person in each of the first four query sets of the design:

1. **QS1, typing errors**: the recorded Latin given name and surname, in either order, with one token edited once (five letters or fewer) or twice (longer): a deletion, a swap of neighbouring letters, or a neighbouring key on a QWERTY keyboard
2. **QS2, diaspora spelling**: the same name respelled by the substitution rules of the experiment manifest (`Hurvitz` → `Hurwitz`, `Serebrinsky` → `Serebrinski`), one applicable rule per token; a person none of whose tokens any rule changes is left out
3. **QS3, Hebrew → Latin**: the recorded Hebrew name, with the person's Hebrew tokens hidden from the archive for that query, so that only their Latin record can be found
4. **QS4, Latin → Hebrew**: the mirror image of QS3

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Query sets QS1–QS4  —  typing errors, respellings and cross-script queries
# ─────────────────────────────────────────────────────────────────────────────
if "persons" not in globals():
    raise RuntimeError("Run the 'Configuration and Helpers' cell first.")

NEIGHBOURS = {   # each letter's neighbouring keys on a QWERTY keyboard
    "q": "wa", "w": "qeas", "e": "wrsd", "r": "etdf", "t": "ryfg", "y": "tugh", "u": "yihj",
    "i": "uojk", "o": "ipkl", "p": "ol", "a": "qwsz", "s": "awedxz", "d": "serfcx",
    "f": "drtgvc", "g": "ftyhbv", "h": "gyujnb", "j": "huikmn", "k": "jiolm", "l": "kop",
    "z": "asx", "x": "zsdc", "c": "xdfv", "v": "cfgb", "b": "vghn", "n": "bhjm", "m": "njk"}
# The manifest's substitution rules: (name, pattern, alternatives). The
# pattern's group is the part replaced by one of the other alternatives.
DIASPORA_RULES = [
    ("sch/sz/sh/ch", r"^(sch|sz|sh|ch)", ["sch", "sz", "sh", "ch"]),
    ("j→ch/kh/h", r"(j)", ["ch", "kh", "h"]),
    ("v↔w", r"(v|w)", ["v", "w"]),
    ("y/i/j initial", r"^(y|i|j)(?=[aeiou])", ["y", "i", "j"]),
    ("tz/ts/z/c", r"(tz|ts|z|c)(?=[ei])", ["tz", "ts", "z", "c"]),
    ("k↔c", r"(k|c)(?=[aou])", ["k", "c"]),
    ("-sky/-ski/-skiy", r"(skiy|sky|ski)$", ["sky", "ski", "skiy"]),
    ("-man/-mann", r"(mann|man)$", ["man", "mann"]),
    ("-berg/-burg", r"(berg|burg)$", ["berg", "burg"]),
    ("-stein/-shtein", r"(shtein|stein)$", ["stein", "shtein"]),
    ("ei/ey/ai", r"(ei|ey|ai)", ["ei", "ey", "ai"]),
    ("u↔oo", r"(oo|u)", ["u", "oo"]),
    ("f↔ph", r"(ph|f)", ["f", "ph"]),
    ("doubled consonant", r"(([bdfgklmnprstz])\2)", None),   # "ss" → "s"
]


def typing_error(word: str, rng: random.Random) -> str:
    """Apply one random typing error (deletion, swap or neighbour key) to `word`."""
    i = rng.randrange(len(word) - 1)
    kind = rng.choice(("delete", "swap", "substitute"))
    if kind == "delete":
        return word[:i] + word[i + 1:]
    if kind == "swap":
        return word[:i] + word[i + 1] + word[i] + word[i + 2:]
    key = unicodedata.normalize("NFKD", word[i])[0].lower()
    replacement = rng.choice(NEIGHBOURS.get(key, "aeiou"))
    return word[:i] + (replacement.upper() if word[i].isupper() else replacement) + word[i + 1:]


def respell(word: str, rng: random.Random) -> tuple[str, str]:
    """Respell `word` with one applicable diaspora rule chosen at random.

    Returns:
        The respelling (in `word`'s capitalisation) and the rule's name, or
        `word` and an empty name when no rule applies.
    """
    lower, options = word.lower(), []
    for name, pattern, alternatives in DIASPORA_RULES:
        for found in re.finditer(pattern, lower):
            start, end = found.span(1)
            for other in (alternatives or [found.group(2)]):
                if other != found.group(1):
                    options.append((lower[:start] + other + lower[end:], name))
    if not options:
        return word, ""
    variant, name = rng.choice(options)
    return (variant.capitalize() if word[:1].isupper() else variant), name


def query_token(display: str, source: dict) -> dict:
    """Return the record of one query token typed for the recorded token `source`."""
    return {"display": display, "norm": normalise(display), "script": token_script(display),
            "field": source["field"], "source_norm": source["norm"]}


def name_tokens(person: dict, script: str) -> list[dict]:
    """Return the person's given-name then surname tokens in `script`."""
    return [t for f in ("given", "surname") for t in person["tokens"]
            if t["field"] == f and t["script"] == script]


def has_full_name(person: dict, script: str) -> bool:
    """Return True if the person has a given name and a surname in `script`."""
    return {t["field"] for t in name_tokens(person, script)} == {"given", "surname"}


def make_query(query_set: str, person: dict, tokens: list[dict], masked: str = "",
               rule: str = "") -> dict:
    """Return one query whose only correct answer is its target person."""
    return {"query_id": f"{query_set}-{person['person_id']}", "split": person["split"],
            "set": query_set, "query_text": " ".join(t["display"] for t in tokens),
            "target_person_id": person["person_id"],
            "relevant_person_ids": json.dumps([person["person_id"]]),
            "masked_columns": masked, "rule": rule,
            "tokens": json.dumps(tokens, ensure_ascii=False)}


def qs1_query(person: dict, rng: random.Random) -> dict:
    """Return the person's typing-error query (QS1)."""
    source = name_tokens(person, "latin")
    if rng.random() < 0.5:
        source = source[::-1]
    words = [t["display"] for t in source]
    chosen = rng.choice([i for i, w in enumerate(words) if len(w) >= 3] or [0])
    for _ in range(1 if len(words[chosen]) <= 5 else 2):
        words[chosen] = typing_error(words[chosen], rng)
    return make_query("QS1", person, [query_token(w, t) for w, t in zip(words, source)])


def qs2_query(person: dict, rng: random.Random) -> dict:
    """Return the person's diaspora-spelling query (QS2), or an empty dictionary
    when no rule applies to any of its tokens (the person is then left out)."""
    source = name_tokens(person, "latin")
    respelled = [respell(t["display"], rng) for t in source]
    if not any(rule for _, rule in respelled):
        return {}
    return make_query("QS2", person, [query_token(w, t) for (w, _), t in zip(respelled, source)],
                      rule="; ".join(rule for _, rule in respelled if rule))


def cross_script_query(query_set: str, person: dict) -> dict:
    """Return the person's Hebrew (QS3) or Latin (QS4) query, own script hidden."""
    script = "hebrew" if query_set == "QS3" else "latin"
    suffix = "he" if script == "hebrew" else "es"
    return make_query(query_set, person, [query_token(t["display"], t)
                                          for t in name_tokens(person, script)],
                      masked=f"given_name_{suffix};surname_{suffix}")


searched = [p for p in persons if p["split"] in ("validation", "test")]
queries = []
for _make in (qs1_query, qs2_query):
    _rng = random.Random(SEED)
    for _person in searched:
        if has_full_name(_person, "latin"):
            _query = _make(_person, _rng)
            if _query:                       # QS2 leaves out a person no rule changes
                queries.append(_query)
for _query_set in ("QS3", "QS4"):
    queries += [cross_script_query(_query_set, p) for p in searched
                if has_full_name(p, "latin") and has_full_name(p, "hebrew")]

print(f"{RULE}\nQuery sets QS1–QS4 (validation and test persons)\n{RULE}")
for _query_set in ("QS1", "QS2", "QS3", "QS4"):
    _examples = [q["query_text"] for q in queries if q["set"] == _query_set][:3]
    print(f"  {_query_set}  {sum(q['set'] == _query_set for q in queries):>4} queries   "
          f"e.g. {' | '.join(_examples)}")
print(f"{RULE}\n  {len(queries)} queries for {len(searched)} persons")

────────────────────────────────────────────────────────────────────────
Query sets QS1–QS4 (validation and test persons)
────────────────────────────────────────────────────────────────────────
  QS1   227 queries   e.g. Valentin rHuvitz | Jose Kfoamn | Sofa Tolcachier
  QS2   195 queries   e.g. Walentin Hoorvitz | Khose Kofmann | Sophia Tolkachier
  QS3   141 queries   e.g. אב הורוויץ | יוסף קאפמאן | שפרה טאלקאטשי
  QS4   141 queries   e.g. Valentin Hurvitz | Jose Kofman | Sofia Tolcachier
────────────────────────────────────────────────────────────────────────
  704 queries for 256 persons


**3. Query Set QS5 and the Query File**

This cell draws the **partial-name** set QS5: `QS5_SIZE` queries sampled from QS1–QS4, each reduced to one token, the surname with probability `QS5_SURNAME_P` and otherwise the given name. A single common name matches many people, so a QS5 query's correct answers are every person in the archive with the same normalised token in the same field and script, plus the target. It writes `./outputs/step_2_queries.csv`.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Query set QS5 and the query file  —  one-token queries, many correct answers
# ─────────────────────────────────────────────────────────────────────────────
if "queries" not in globals():
    raise RuntimeError("Run the 'Query Sets QS1–QS4' cell first.")


def token_owners() -> dict[tuple, set[str]]:
    """Map each (field, script, norm) to the persons holding such a token."""
    owners = defaultdict(set)
    for person in persons:
        for token in person["tokens"]:
            owners[(token["field"], token["script"], token["norm"])].add(person["person_id"])
    return owners


def qs5_query(parent: dict, rng: random.Random, owners: dict) -> dict:
    """Reduce `parent` to one token and list every person it correctly finds."""
    field_name = "surname" if rng.random() < QS5_SURNAME_P else "given"
    in_field = [t for t in json.loads(parent["tokens"]) if t["field"] == field_name]
    kept = in_field[-1] if field_name == "surname" else in_field[0]
    target = persons_by_id[parent["target_person_id"]]
    script = next(t["script"] for t in target["tokens"]
                  if t["field"] == field_name and t["norm"] == kept["source_norm"])
    relevant = owners[(field_name, script, kept["source_norm"])] | {target["person_id"]}
    return parent | {"query_id": f"QS5-{parent['query_id']}", "set": "QS5",
                     "query_text": kept["display"], "rule": "",
                     "relevant_person_ids": json.dumps(sorted(relevant)),
                     "tokens": json.dumps([kept], ensure_ascii=False)}


_rng = random.Random(SEED)
_parents = list(queries)
_rng.shuffle(_parents)
_owners = token_owners()
queries += [qs5_query(parent, _rng, _owners) for parent in _parents[:QS5_SIZE]]
write_csv(OUTPUTS_DIR / "step_2_queries.csv", queries)
query_counts = Counter((q["split"], q["set"]) for q in queries)

print(f"{RULE}\nQuery file: {OUTPUTS_DIR}/step_2_queries.csv\n{RULE}")
for _query_set in ("QS1", "QS2", "QS3", "QS4", "QS5"):
    print(f"  {_query_set}  validation {query_counts[('validation', _query_set)]:>4}   "
          f"test {query_counts[('test', _query_set)]:>4}")
print(f"{RULE}\n  {len(queries)} queries")

────────────────────────────────────────────────────────────────────────
Query file: outputs/step_2_queries.csv
────────────────────────────────────────────────────────────────────────
  QS1  validation  112   test  115
  QS2  validation   94   test  101
  QS3  validation   77   test   64
  QS4  validation   77   test   64
  QS5  validation  153   test  147
────────────────────────────────────────────────────────────────────────
  1004 queries


**4. Consistency Check**

This cell reads the query file back and checks that every target is a validation or test person, that every masked column belongs to the target, and that every QS5 query lists its target among its correct answers.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the query file against the persons
# ─────────────────────────────────────────────────────────────────────────────
if "query_counts" not in globals():
    raise RuntimeError("Run every cell above first.")

with (OUTPUTS_DIR / "step_2_queries.csv").open(encoding="utf-8", newline="") as _file:
    _stored = list(csv.DictReader(_file))
_problems = []
for _query in _stored:
    _target = persons_by_id.get(_query["target_person_id"])
    if _target is None or _target["split"] != _query["split"] or _query["split"] == "train":
        _problems.append(f"{_query['query_id']}: target not a {_query['split']} person")
    elif not set(filter(None, _query["masked_columns"].split(";"))) <= {
            t["column"] for t in _target["tokens"]}:
        _problems.append(f"{_query['query_id']}: masks a column its target lacks")
    elif _target["person_id"] not in json.loads(_query["relevant_person_ids"]):
        _problems.append(f"{_query['query_id']}: target not among the correct answers")
print(f"{RULE}\nConsistency: {OUTPUTS_DIR}/step_2_queries.csv\n{RULE}")
for _problem in _problems[:10]:
    print(f"  !! {_problem}")
print(f"  {'OK' if not _problems else '!!'} {len(_stored)} queries checked")
_split_counts = Counter(q["split"] for q in queries)
print(f"\n✓ Query sets complete ({_split_counts['validation']} validation and "
      f"{_split_counts['test']} test queries).")
print(f"  consistent: {not _problems}")

────────────────────────────────────────────────────────────────────────
Consistency: outputs/step_2_queries.csv
────────────────────────────────────────────────────────────────────────
  OK 1004 queries checked

✓ Query sets complete (513 validation and 491 test queries).
  consistent: True
